# M1A3 — Orquestrando um Agente ReAct com GPT-4o-mini
Nesta aula exploramos, passo a passo, como estruturar um agente simples seguindo o padrão **ReAct (Reasoning + Acting)**. Cada bloco de código abaixo cumpre um estágio específico do fluxo: preparar dependências, configurar o cliente da API, definir o comportamento do agente e testar ciclos de pensamento/ação/observação.

# Tic em Trilhas - Construção de Agentes Inteligentes com IA
## Módulo 1 - Aula 3


### Preparando dependências e variáveis de ambiente
Antes de interagir com qualquer modelo, carregamos bibliotecas de apoio (como `dotenv` para ler o `.env`) e instanciamos o cliente oficial da OpenAI. Esse setup garante que as chaves e o contexto de rede estejam disponíveis para o restante da aula.

In [ ]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

In [58]:
client = OpenAI()

### Smoke test do modelo
Executamos uma primeira chamada ao `gpt-4o-mini` apenas com uma saudação. Essa etapa serve para validar se as credenciais e a conexão HTTP estão corretas antes de evoluir para estruturas de agente mais complexas.

In [59]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Bom dia, como vai?"}],
    temperature=0
)

print(response.choices[0].message.content)

Bom dia! Estou aqui e pronto para ajudar. Como posso assisti-lo hoje?


### Estrutura base do agente
Aqui definimos uma classe simples que armazena o estado da conversa e injeta a instrução de sistema apenas uma vez. Ao chamar a instância com novas mensagens, o agente delega à API da OpenAI, atualiza o histórico e devolve a última resposta — exatamente o padrão ReAct reduzido.

### Instruções de sistema no estilo ReAct
O prompt de sistema descreve o ciclo **Pensamento → Ação → PAUSA → Observação → Resposta** e documenta quais ferramentas o agente pode chamar por nome. Essa engenharia de prompt “programa” o modelo para produzir saídas estruturadas que conseguimos interpretar no código.

In [60]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gpt-4o-mini", 
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content
    

### Instanciando o agente com o prompt ReAct
Criamos a instância `campeao` com o prompt anterior, garantindo que toda nova conversa já comece com as regras de pensamento/ação disponíveis.

### Simulando um ciclo completo manualmente
Rodamos `campeao` com uma pergunta simples para observar como o modelo gera **Pensamento/Ação/PAUSA**. Depois, imitamos o retorno de uma ferramenta (preço da Moqueca) e mostramos ao agente via `Observation`, forçando-o a concluir com a `Resposta` final do ciclo.

### Definindo ferramentas disponíveis
A lista de `known_actions` funciona como o inventário de ferramentas que o agente pode acionar durante o ciclo ReAct. Cada item associa um nome textual a uma função Python concreta, permitindo que o modelo “planeje” ações em linguagem natural e o ambiente execute-as de fato.

In [61]:
prompt = """
Você executa em um ciclo de Pensamento, Ação, PAUSA, Observação.
No final do ciclo você fornece uma Resposta
Use Pensamento para descrever seus pensamentos sobre a pergunta que foi feita.
Use Ação para executar uma das ações disponíveis - então retorne PAUSA.
Observação será o resultado da execução dessas ações.

Suas ações disponíveis são:

calcular:
ex: calcular: 4 * 7 / 3
Executa um cálculo e retorna o número - usa Python então certifique-se de usar sintaxe de ponto flutuante se necessário

preco_prato:
ex: preco_prato: Feijoada
retorna o preço do prato quando fornecido o nome

Exemplo de sessão:

Pergunta: Quanto custa uma Moqueca?
Pensamento: Devo verificar o preço da Moqueca usando preco_prato
Ação: preco_prato: Moqueca
PAUSA

Você será chamado novamente com isto:

Observação: Uma Moqueca custa R$ 89,90

Você então fornece:

Resposta: Uma Moqueca custa R$ 89,90
""".strip()

In [62]:
def calculate(formula):
    return eval(formula)

def preco_prato(nome):
    if nome == "Feijoada":
        return "Uma Feijoada custa R$ 75,90"
    elif nome == "Moqueca":
        return "Uma Moqueca custa R$ 89,90"
    elif nome == "Picanha":
        return "Uma Picanha custa R$ 129,90"
    else:
        return "Prato não encontrado no cardápio"

known_actions = {
    "calculate": calculate,
    "preco_prato": preco_prato
}

In [63]:
campeao = Agent(prompt)

In [64]:
result = campeao("Quanto custa uma Moqueca?")
print(result)

Pensamento: Devo verificar o preço da Moqueca usando preco_prato.  
Ação: preco_prato: Moqueca  
PAUSA


In [72]:
result = preco_prato("Moqueca")
print(result)

Uma Moqueca custa R$ 89,90


In [66]:
next_prompt = "Observation: {}".format(result)

In [73]:
campeao(next_prompt)

'Resposta: Uma Moqueca custa R$ 89,90.'

In [74]:
campeao.messages

[{'role': 'system',
  'content': 'Você executa em um ciclo de Pensamento, Ação, PAUSA, Observação.\nNo final do ciclo você fornece uma Resposta\nUse Pensamento para descrever seus pensamentos sobre a pergunta que foi feita.\nUse Ação para executar uma das ações disponíveis - então retorne PAUSA.\nObservação será o resultado da execução dessas ações.\n\nSuas ações disponíveis são:\n\ncalcular:\nex: calcular: 4 * 7 / 3\nExecuta um cálculo e retorna o número - usa Python então certifique-se de usar sintaxe de ponto flutuante se necessário\n\npreco_prato:\nex: preco_prato: Feijoada\nretorna o preço do prato quando fornecido o nome\n\nExemplo de sessão:\n\nPergunta: Quanto custa uma Moqueca?\nPensamento: Devo verificar o preço da Moqueca usando preco_prato\nAção: preco_prato: Moqueca\nPAUSA\n\nVocê será chamado novamente com isto:\n\nObservação: Uma Moqueca custa R$ 89,90\n\nVocê então fornece:\n\nResposta: Uma Moqueca custa R$ 89,90'},
 {'role': 'user', 'content': 'Quanto custa uma Moqueca

### Adicionar um loop   

In [ ]:
action_re = re.compile('^Ação: (\\w+): (.*)$')

In [75]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        print(actions)
        if actions:
            # Há uma ação para executar
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Ação desconhecida: {}: {}".format(action, action_input))
            print(" -- executando {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observação:", observation)
            next_prompt = "Observação: {}".format(observation)
        else:
            return

In [77]:
question = """Tenho 2 pratos, um Feijoada e uma Picanha. \
Qual é o custo total dos dois pratos?"""
query(question)

Pensamento: Primeiro, preciso verificar o preço da Feijoada e da Picanha usando a ação preco_prato. Depois, somarei os dois preços para encontrar o custo total. 

Ação: preco_prato: Feijoada
PAUSA
[<re.Match object; span=(0, 27), match='Ação: preco_prato: Feijoada'>]
 -- executando preco_prato Feijoada
Observação: Uma Feijoada custa R$ 75,90
Ação: preco_prato: Picanha
PAUSA
[<re.Match object; span=(0, 26), match='Ação: preco_prato: Picanha'>]
 -- executando preco_prato Picanha
Observação: Uma Picanha custa R$ 129,90
Resposta: O custo total dos dois pratos, uma Feijoada e uma Picanha, é R$ 205,80.
[]
